# Etap 4 i 5: Podział danych, Walidacja i Modelowanie (Regresja)
W tym notatniku przeprowadzimy podział danych, zdefiniujemy strategię walidacji oraz zbudujemy modele regresyjne dla zmiennej docelowej `SalaryUSD_Log`.

In [9]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn modules for splitting and validation
from sklearn.model_selection import train_test_split, KFold

# Set plotting style
sns.set_theme(style="whitegrid")

# Load the processed data from the previous stage
file_path = '../data/processed/processed_data.csv'
df = pd.read_csv(file_path)

print(f"Processed data loaded successfully. Shape: {df.shape}")

Processed data loaded successfully. Shape: (14755, 2952)


### 4. Podział danych i walidacja

**Uzasadnienie wyboru parametrów:**
1. **Zbiór treningowy i testowy (Train/Test Split):** Zastosowano podział w proporcji **80/20** (`test_size=0.2`). Nasz przygotowany zbiór danych posiada blisko 14 900 rekordów. Odłożenie 20% (około 3000 obserwacji) jako zbioru testowego jest w pełni wystarczające do miarodajnej i statystycznie istotnej oceny modelu na danych niewidzianych podczas treningu. Pozostałe 80% to odpowiednia ilość danych, by złożone algorytmy (np. Random Forest) mogły wychwycić wzorce bez ryzyka niedouczenia. Zastosowano `random_state=42` dla zapewnienia powtarzalności wyników.
2. **Walidacja krzyżowa (K-Fold Cross-Validation):** Wybrano strategię **k=5** ze włączonym tasowaniem (`shuffle=True`). Jest to standard branżowy, który stanowi optymalny kompromis między wariancją estymatora błędu a czasem obliczeń. K-fold upewnia nas, że model faktycznie generalizuje wiedzę, a nie tylko zapamiętuje specyficzny układ danych, co jest szczególnie ważne przy tak dużej liczbie cech (ponad 2900 kolumn po One-Hot Encoding).

In [10]:
# 1. Define Target and Features
# Our target is the logarithmically transformed salary
y = df['SalaryUSD_Log']

# We must drop both the raw/log salary AND the classification target from our features (X)
career_col = [col for col in df.columns if 'CareerPlans' in col][0]
X = df.drop(columns=['SalaryUSD', 'SalaryUSD_Log', career_col])

# 2. Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Testing set: X={X_test.shape}, y={y_test.shape}")

# 3. Define K-Fold Cross Validation strategy for model evaluation and hyperparameter tuning
# We use shuffle=True to ensure random distribution of categories across folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"Cross-validation strategy defined: {kf.get_n_splits()}-Fold CV")

Training set: X=(11804, 2949), y=(11804,)
Testing set: X=(2951, 2949), y=(2951,)
Cross-validation strategy defined: 5-Fold CV


### 5. Budowa modeli uczenia maszynowego (Regresja)

Zgodnie z wymaganiami projektu, zbudowałam trzy różne modele regresyjne, aby przewidzieć zarobki (`SalaryUSD_Log`):
1. **Ridge Regression** jako model bazowy (baseline).
2. **Random Forest Regressor** jako model nieliniowy, dobrze radzący sobie z dużą ilością cech kategorycznych.
3. **XGBoost Regressor** z optymalizacją hiperparametrów, jako zaawansowany algorytm gradient boosting.

Do oceny jakości modeli wykorzystałam metryki MAE (Mean Absolute Error) oraz $R^2$. Proces walidacji oparłam na zdefiniowanej wcześniej 5-krotnej walidacji krzyżowej (K-Fold CV). Wyniki błędu logarytmicznego są transformowane z powrotem do realnych wartości w dolarach (USD) dla ułatwienia interpretacji.

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

# Dictionary to store results for final comparison
model_results = {}

# Helper function to evaluate and print model metrics
def evaluate_regression_model(model_name, model, X_train, y_train, X_test, y_test, kfold_obj):
    # Fit the model
    model.fit(X_train, y_train)
    
    # Make predictions
    preds = model.predict(X_test)
    
    # Calculate metrics on test set
    # Reversing the log transformation using expm1 for interpretability
    mae_usd = mean_absolute_error(np.expm1(y_test), np.expm1(preds))
    rmse_usd = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(preds)))
    r2 = r2_score(y_test, preds)
    
    # Perform Cross-Validation on the training set
    # We use negative mean absolute error for scoring in cross_val_score
    cv_scores = cross_val_score(model, X_train, y_train, cv=kfold_obj, scoring='neg_mean_absolute_error', n_jobs=-1)
    
    # Transform CV scores back to normal MAE space
    cv_mae_log = -cv_scores.mean()
    
    print(f"--- {model_name} ---")
    print(f"Test MAE: ${mae_usd:,.2f}")
    print(f"Test RMSE: ${rmse_usd:,.2f}")
    print(f"Test R^2: {r2:.4f}")
    print(f"Cross-Validation MAE (Log space mean): {cv_mae_log:.4f}")
    print("-" * 30)
    
    # Save results to dictionary
    model_results[model_name] = {
        'MAE_USD': mae_usd,
        'RMSE_USD': rmse_usd,
        'R2': r2,
        'CV_MAE_Log': cv_mae_log
    }
    
    return model

In [12]:
from sklearn.linear_model import Ridge

print("Training Baseline Model (Ridge Regression)...")
# Initialize the model with alpha=10.0 to handle the high dimensionality (2900+ columns)
ridge_model = Ridge(alpha=10.0, random_state=42)

# Evaluate the model using our custom function
evaluate_regression_model("Ridge Baseline", ridge_model, X_train, y_train, X_test, y_test, kf)

Training Baseline Model (Ridge Regression)...
--- Ridge Baseline ---
Test MAE: $22,903.44
Test RMSE: $33,357.65
Test R^2: 0.4873
Cross-Validation MAE (Log space mean): 0.2646
------------------------------


,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",10.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",42


In [13]:
from sklearn.ensemble import RandomForestRegressor

print("Training Random Forest Regressor (This might take a minute)...")
# Using n_jobs=-1 to use all CPU cores and speed up the training process
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Evaluate the model
evaluate_regression_model("Random Forest", rf_model, X_train, y_train, X_test, y_test, kf)

Training Random Forest Regressor (This might take a minute)...
--- Random Forest ---
Test MAE: $20,743.78
Test RMSE: $31,289.44
Test R^2: 0.5520
Cross-Validation MAE (Log space mean): 0.2444
------------------------------


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [15]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
import re
print("Starting Hyperparameter Tuning for XGBoost...")

# 1. Define the hyperparameter grid to search through
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [5, 7, 9],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

# 2. Initialize the base XGBoost model
xgb_base = XGBRegressor(random_state=42, n_jobs=-1)

# 3. Setup Randomized Search (using 3-fold CV for faster tuning during development)
xgb_random = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=5, # Tries 5 random combinations from the grid
    scoring='neg_mean_absolute_error',
    cv=3, 
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 4. Fit the search model to find the best parameters
X_train.columns = [re.sub(r'[\[\]<>]', '_', str(col)) for col in X_train.columns]
X_test.columns = [re.sub(r'[\[\]<>]', '_', str(col)) for col in X_test.columns]
xgb_random.fit(X_train, y_train)

print(f"\nBest XGBoost parameters found: {xgb_random.best_params_}\n")

# 5. Evaluate the best version of XGBoost using our custom function
best_xgb = xgb_random.best_estimator_
evaluate_regression_model("Tuned XGBoost", best_xgb, X_train, y_train, X_test, y_test, kf)

Starting Hyperparameter Tuning for XGBoost...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

Best XGBoost parameters found: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.05}

--- Tuned XGBoost ---
Test MAE: $20,516.64
Test RMSE: $30,894.58
Test R^2: 0.5725
Cross-Validation MAE (Log space mean): 0.2389
------------------------------


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes